# arm-gym GRPO Training
ARM AArch64 assembly superoptimizer trained with GRPO. Beats gcc -O3 on LLVM-MCA cycle estimates.

**Setup**: Add dataset `im.vetri0/arm-gym-code` to this notebook (+ button in right panel)

**Accelerator**: T4 x2 for smoke, L4 x4 for full run

**Internet**: ON required (model download + LLVM 21 apt)

**Expected time**: ~60 min for 200-step single GPU run

In [ ]:
# Cell 1: copy from dataset input + install
import subprocess, os, sys, shutil, glob

WORKDIR = '/kaggle/working/arm-gym'

# Find the dataset in /kaggle/input/
# Adjust this path if your dataset slug differs
candidates = glob.glob('/kaggle/input/*/arm_gym') + glob.glob('/kaggle/input/arm-gym*') + glob.glob('/kaggle/input/*/arm-gym')
INPUT_DIR = candidates[0] if candidates else '/kaggle/input/arm-gym-code'
print(f'Input dir: {INPUT_DIR}')

if not os.path.exists(WORKDIR):
    shutil.copytree(INPUT_DIR, WORKDIR)
    print(f'Copied {INPUT_DIR} -> {WORKDIR}')
else:
    print('WORKDIR already exists, skipping copy')

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print(f'cwd: {os.getcwd()}')

r = subprocess.run(['bash', 'kaggle/setup.sh'], capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('setup.sh failed')

In [ ]:
# Cell 2: smoke - validates toolchain + dataset + reward loop (~5 min)
import subprocess
r = subprocess.run(['python', 'kaggle/train.py', '--smoke'], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('smoke failed')
print('smoke OK')

In [ ]:
# Cell 3: detect GPU stack
import subprocess
r = subprocess.run(['python', 'scripts/smoke_4xl4.py'], capture_output=True, text=True)
print(r.stdout)

verdict = 'single_gpu'
for line in r.stdout.splitlines():
    if line.startswith('verdict:'):
        verdict = line.split(':', 1)[1].strip()
print(f'Stack: {verdict}')

In [ ]:
# Cell 4: training run (~60 min on T4, ~25 min on L4)
# 200 steps, scalar kernels (difficulty 1), lora-rank 8 for speed
import subprocess

r = subprocess.run([
    'python', 'kaggle/train.py',
    '--stack', 'single_gpu',
    '--steps', '200',
    '--max-train', '128',
    '--num-generations', '4',
    '--lora-rank', '8',
    '--difficulty-max', '1',
    '--out', 'runs/short',
], capture_output=True, text=True)

print(r.stdout[-5000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-3000:])

In [ ]:
# Cell 4b: 4xL4 DDP run (400 steps) - ONLY run if Cell 3 shows 4 GPUs
# Uncomment everything below to execute
import subprocess

# r = subprocess.run([
#     'accelerate', 'launch', '--multi_gpu', '--num_processes', '4',
#     'kaggle/train.py',
#     '--stack', 'plain_trl_ddp',
#     '--steps', '400',
#     '--max-train', '256',
#     '--num-generations', '4',
#     '--out', 'runs/4xl4',
# ], capture_output=True, text=True)
# print(r.stdout[-5000:])
# if r.returncode != 0:
#     print('STDERR:', r.stderr[-3000:])

print('Uncomment above to run 4xL4 DDP training')

In [ ]:
# Cell 5: plots + display
import subprocess, glob
from IPython.display import Image, display

# use whichever run completed
log_path = 'runs/short/log.csv'
out_path = 'artifacts/plots'

r = subprocess.run(
    ['python', 'kaggle/plot_curves.py', '--log', log_path, '--out', out_path],
    capture_output=True, text=True
)
print(r.stdout)
if r.returncode != 0:
    print('plot error:', r.stderr)

for png in sorted(glob.glob(f'{out_path}/*.png')):
    print(png)
    display(Image(filename=png))

In [ ]:
# Cell 6: zip and download
import subprocess
from IPython.display import FileLink
subprocess.run(['zip', '-r', 'runs.zip', 'runs/', 'artifacts/'], check=False)
FileLink('runs.zip')